# Smoke Test of the Pilot Models

This notebook runs the smoke test of the pilot experiment on Kaggle. It trains a small WIND model and a small SpherIDiff model for a short time under identical settings, and it records how the training loss and the forecast error change. The runs use the ERA5 cache at 5.6 degrees, attached as a Kaggle input, and the repository code, cloned from GitHub. The two models run at the same time, each on its own GPU. With one GPU, they run one after the other. The notebook is designed for a background run, in which Kaggle executes the whole notebook and keeps the files of `/kaggle/working` when it ends.

The smoke test has six stages for each model:

- **static**: the output shape, the zero start of the output, and the parameter count.
- **overfit**: training on 8 fixed windows, where the loss must fall clearly.
- **short**: a training run of 2,000 steps with evaluations at equal intervals.
- **resume**: a stop and resume of a short run, compared with an uninterrupted run.
- **tasks**: forecasting and the three guided reconstruction tasks on two windows, to confirm that every code path runs.
- **timing**: an estimate of the training time of the full pilot run, from the measured speed.

Everything that the notebook produces is packed into `smoke_results.zip` in the working folder.

In [ ]:
import os
import sys
import time
from pathlib import Path

import torch

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
RUNS_ROOT = WORK_ROOT / "runs"
SESSION_HOURS = 9.0
PACKAGING_RESERVE_MINUTES = 20
SMOKE_EXPERIMENTS = ["smoke_wind_s", "smoke_spheridiff_b4"]
ARCHIVE = WORK_ROOT / "smoke_results.zip"

## Inputs and hardware

The repository code is cloned from GitHub (`https://github.com/syeddaniyalg/spheri-diff`) into the working folder, which requires internet access for the notebook. The data cache is read from the attached dataset `thesyeddaniyal/era5-pilot-6degrees`, at `/kaggle/input/datasets/thesyeddaniyal/era5-pilot-6degrees`. The folder that holds `stats.npz` is used as the cache root. When that input path does not exist, the whole of `/kaggle/input` is searched for `stats.npz`.

In [ ]:
import subprocess

REPOSITORY_URL = "https://github.com/syeddaniyalg/spheri-diff"
CODE_ROOT = WORK_ROOT / "spheri-diff"
DATASET_ROOT = INPUT_ROOT / "datasets" / "thesyeddaniyal" / "era5-pilot-6degrees"


def locate(marker, root):
    for current, _, files in os.walk(root):
        if marker in files:
            return Path(current)
    raise FileNotFoundError(f"{marker} was not found in {root}")


if not CODE_ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(CODE_ROOT)], check=True)
CACHE_ROOT = locate("stats.npz", DATASET_ROOT if DATASET_ROOT.exists() else INPUT_ROOT)
sys.path.insert(0, str(CODE_ROOT / "src"))
print("code:", CODE_ROOT)
print("data:", CACHE_ROOT)
for index in range(torch.cuda.device_count()):
    print("gpu", index, torch.cuda.get_device_name(index))

## Running the jobs

Each job is a separate process with its own log file in `runs/_logs`, and the processes of different GPUs run at the same time. The notebook prints every line that a job writes as soon as it appears, prefixed with the elapsed time, the GPU and the job name, so that all jobs can be followed in one output. The same lines are kept in `runs/_logs/parallel.log`.

The jobs and the packing of the results share one cell with a `try` and a `finally`. When a job fails, or the cell is interrupted, the figures, the status file and the archive are still produced. The archive is also refreshed every 30 minutes during the run. The time limit of the session is set by `SESSION_HOURS`, and the jobs are told to stop with a checkpoint before it, leaving time for packing.

In [ ]:
from spheridiff.kaggle.finalise import finalise, refresh_archive
from spheridiff.kaggle.jobs import assign_queues, smoke_job
from spheridiff.kaggle.runner import ParallelRunner, RunContext

DEADLINE = time.time() + SESSION_HOURS * 3600 - PACKAGING_RESERVE_MINUTES * 60
GROUPS = [[smoke_job(name)] for name in SMOKE_EXPERIMENTS]
QUEUES = assign_queues(GROUPS, torch.cuda.device_count())
CONTEXT = RunContext(CODE_ROOT, RUNS_ROOT, RUNS_ROOT / "_logs", CACHE_ROOT, DEADLINE)

COMPARISONS = [("smoke_spheridiff_b4", "smoke_wind_s")]

RUNNER = None
try:
    RUNNER = ParallelRunner(QUEUES, CONTEXT, refresh=lambda: refresh_archive(RUNS_ROOT, ARCHIVE))
    RUNNER.run()
finally:
    OUTCOMES = RUNNER.outcomes() if RUNNER else []
    finalise(RUNS_ROOT, ARCHIVE, OUTCOMES, SMOKE_EXPERIMENTS, references=[], comparisons=COMPARISONS)
print(OUTCOMES)
print(ARCHIVE, f"{ARCHIVE.stat().st_size / 1024**2:.1f} MB")

## Results

The table lists the outcome of every stage for each model. The `timing` stage reports the measured samples per second and the estimated hours of the pilot run, with and without a 20% margin. The figures show the validation loss, the normalised forecast error and the normalised CRPS against steps, samples seen and training time, and the training loss smoothed over 50 steps. The comparison summary lists, for each SpherIDiff run against each WIND run, the step at which SpherIDiff reaches the final error of WIND and stays there, the errors at fixed fractions of training, the mean error over the whole curve, and a paired difference with a 95% interval over the evaluation windows.

In [ ]:
import json

for name in SMOKE_EXPERIMENTS:
    path = RUNS_ROOT / name / "smoke_report.json"
    if not path.exists():
        print(name, "wrote no report, see runs/_logs")
        continue
    report = json.loads(path.read_text())
    print(name, "passed" if report["passed"] else "has failures")
    for stage in ["static", "overfit", "short", "resume", "tasks", "timing"]:
        print(f"  {stage:8s}", report[stage])

summary = RUNS_ROOT / "comparison_summary.txt"
if summary.exists():
    print(summary.read_text(encoding="utf-8"))

In [ ]:
from IPython.display import Image, display

for figure in ["evaluation_curves.png", "training_loss.png"]:
    path = RUNS_ROOT / figure
    if path.exists():
        display(Image(filename=str(path)))